# **Data Visualisation (ST1502) CA1: Data Cleaning (Provided Dataset)**
Name: Goh Kun Ming

Class: DAAA/FT/2B/02

Admin Number: P2415691

Lecturer: Senior Lecturer Peter Wai Tong Lee

---
# 1. Environment Setup and Libraries Import

In this section, we will be importing the various relevant libraries required to clean the provided Data. We will also need to mount the Google Drive so as to allow for quick and efficient access to the provided dataset.

In [ ]:
"""DAVI CA1 — Basic Colab Setup & Utilities.

This module sets up display options, mounts Google Drive, and provides
lightweight styling helpers for DataFrame rendering in notebooks.
It also initialises project paths for raw and cleaned datasets.

Usage:
    • Run this cell once at the start of your session.
    • Use `RAW_CSV` and `CLEAN_CSV` as your canonical file paths.

Author: Goh Kun Ming
"""

# =========================
# Import Libraries
# =========================
from pathlib import Path
from typing import Optional, Any

import plotly.express as px
import plotly.io as pio
import pandas as pd
import numpy as np
import re
import os
import pandas as pd
import numpy as np
import matplotlib as mpl
try:
    from IPython.display import display
except ImportError:
    def display(obj: Any) -> None:
        print(obj)


# =========================
# Project Paths
# =========================
BASE_DIR: Path = Path(__file__).resolve().parents[1] if "__file__" in globals() else Path("./")
BASE_DIR.mkdir(parents=True, exist_ok=True)

RAW_CSV: Path = BASE_DIR / "data/raw/hdb_resale_flat_prices_2017_2025_raw.csv"
CLEAN_CSV: Path = BASE_DIR / "data/processed/hdb_resale_flat_prices_clean.csv"

With reference to the Code Cell above, we are able to confirm that the libraries and the Google Drive have been successfully imported and mounted and we are able to proceed to conduct the Data Cleaning process on the provided dataset.

---
# 2. Data Loading & Column Normalisation

In this section, we will be loading the provided HDB dataset into the environemnt in a dataframe. Subsequently, we will be normalising all the columns to a standardised format so as to ensure consistency throughout the features within the data. Finally, we will be previewing the dataframe to get a better sensing and understanding of what the dataset holds with reference to the contents each of the data column holds.

In [ ]:
"""DAVI CA1 — Data Loading & Column Normalisation.

Loads the raw HDB Resale Flat Prices dataset, standardises all column names
to snake_case, and displays a styled preview (top 10 rows) with a dark header
and blue gradient for dark-mode readability.
"""

# =========================
# Load Raw Dataset
# =========================
df = pd.read_csv(RAW_CSV)


# =========================
# Standardise Column Names
# =========================
def normalise_colname(s: str) -> str:
    """Convert a raw column name to lowercase snake_case.

    Removes whitespace, replaces spaces and hyphens with underscores,
    and lowercases all characters.

    Args:
        s: Raw column name.

    Returns:
        Standardised column name suitable for analysis.
    """
    return s.strip().lower().replace(" ", "_").replace("-", "_")


df.columns = [normalise_colname(c) for c in df.columns]


# =========================
# Display Preview
# =========================
def style_blue(df_in):
    """Return a styled DataFrame preview with a dark header.

    Applies a dark header background and a blue gradient on cells
    for better visibility in dark-themed environments.

    Args:
        df_in: DataFrame to style.

    Returns:
        A pandas Styler object for notebook display.
    """
    return (
        df_in.style.background_gradient(cmap="Blues", axis=None)
        .set_properties(**{
            "border": "1px solid #444",
            "font-size": "12px",
            "color": "#e8e8e8",
        })
        .set_table_styles([
            {
                "selector": "th",
                "props": [
                    ("background-color", "#1f2c3b"),
                    ("color", "#f1f1f1"),
                    ("border", "1px solid #555"),
                    ("font-weight", "bold")
                ]
            },
            {
                "selector": "td",
                "props": [
                    ("background-color", "#101820"),
                ]
            }
        ])
    )


# =========================
# Display Dataframe
# =========================
display(style_blue(df.head(10)))

With reference to the Code Cell above, we are able to verify that the dataset have been successfully imported into the environment. We are also able to make a few observations within the dataset with reference to the metadata provided. The observations are indicated below

**month**
- Metadata Reference:
> Given in the format of year-month. You may retrieve the year data from this column, which may be useful when analysing the time trend for HDB resale price.

- No inconsistencies observed.

- A potential feature engineering opportunitiy is observed where we are able to split the year and month so as to potentially conduct Time Series Analysis / Forecast.

**town**
- Metadata Reference:
> Town location should be one of the key factors affecting HDB resale price — you are generally expecting an HDB flat in Orchard has a much higher resale price than Yishun given the same flat type.

- Inconsistencies observed where there are both 'AMK' and 'ANG MO KIO'. There are most likely more inconsistencies like these which we will need to identify and address during the data cleaning stage.

- A data format standardisation approach may be required to upkeep the quality of the column where we change the current data entry format of full capitalisation to a data entry format of first-letter capitalisation (i.e., 'ANG MO KIO' to 'Ang Mo Kio').

**flat_type**
- Metadata Reference:
> There are 7 different kinds of flat types: 1 Room, 2 Room, 3 Room, 4 Room, 5 Room, EC and Multi-generation. Among which the 4 Room HDB flats are the most popular ones in Singapore.

- Inconsistencies observed where there are both '3-ROOM' and '3 ROOM'. There are most likely more inconsistencies like these which we will need to identify and address during the data cleaning stage.

- A data format standardisation approach may be required to upkeep the quality of the column where we change the current data entry format of full capitalisation to a data entry format of first-letter capitalisation (i.e., '3 ROOM' to '3 Room').

**block**
- Metadata Reference:
> The block number of the flat.

- Inconsistencies observed where there are block numbers with special characters such as '@@230' and '@406'. This is most likely due to data entry errors which we will need to indentify and address during the data cleaning stage.

**street_name**
- Metadata Reference:
> The street name where the flat is located.

- Inconsistencies observed where there are both 'AMK' and 'ANG MO KIO'. There are most likely more inconsistencies like these which we will need to identify and address during the data cleaning stage.

- A data format standardisation approach may be required to upkeep the quality of the column where we change the current data entry format of full capitalisation to a data entry format of first-letter capitalisation (i.e., 'ANG MO KIO' to 'Ang Mo Kio').

- During the Data Cleaning segment, we should also approach the issues of short-form used despite it being a common and standard way of writing address in Singapore (i.e., 'AVE' to 'Avenue').

**storey_range**
- Metadata Reference:
> The range of storeys where the flat is located. This column is given as a string rather than numbers.

- Inconsistencies observed where there are block numbers with special characters such as '01@03' and '10@12'. This is most likely due to data entry errors which we will need to indentify and address during the data cleaning stage.

- A data format standardisation approach may be required to upkeep the quality of the column where we change the current data entry format of full capitalisation to a data entry format of no capitalisation (i.e., 'TO' to 'to').

- A potential feature engineering opportunitiy is observed where we are able to split the starting storey and the last storey to conduct independent analysis.

**floor_area_sqm**
- Metadata Reference:
> The floor area of the flat in square meters.

- No inconsistencies observed.

**flat_model**
- Metadata Reference:
> There are different flat models. This factor would play a key role in the overall flat price. E.g., the DBSS (Design, Build and Sell Scheme) flats would have a higher resale price considering it allows buyers to design the HDB based on their own style.

- No inconsistencies observed.

- A unique check will be required to verify and cross check the content to the expected values.

**lease_commence_date**
- Metadata Reference:
> The year the lease commenced

- No inconsistencies observed.

**remaining_lease**
- Metadata Reference:
> Singapore HDB has a lease of 99 years. This column data has quite a few NULL values, and it is calculated based on different years. You may need to adjust this column data accordingly when building the model.

- Data standardisation will need to be applied as there are various entries which utilises short forms instead of the full form (i.e., 62 y 05 months).

- Data entry which contains no months should have '0 months' instead of leaving the month field empty entirely.

- A data format standardisation approach may be required to upkeep the quality of the column where we change the current data entry format of no capitalisation to a data entry format of first-letter capitalisation (i.e., 'years' to 'Years').

**resale_price**
- Metadata Reference:
> The resale price of the flat.

- No inconsistencies observed.

With the observations indicated above, we will proceed to conduct the Missing Values check in the next section.

---
# 3. Missing Values Identification

In this section, we will be indentifying any missing values in the HDB dataset provided. Should there be any missing values identified, we will be required to subsequently identify a suitable approach in handling the missing values depending on the scale of the missing values and what we hope to achieve through the imputation. The identification process will be conducted in the code cell below.

In [ ]:
"""DAVI CA1 — Column-wise Missingness Overview.

Computes and visualises the proportion of missing data per column in the
HDB Resale Flat Prices dataset.
Each row is coloured dark red if the column has missing data, or dark green
if the column is fully complete.
"""

# =========================
# Column-wise Missingness
# =========================
miss_df = (
    pd.DataFrame({
        "column": df.columns,
        "Missing Data Count": df.isna().sum().values,
        "Percentage Missing (%)": (df.isna().mean().values * 100).round(2),
    })
    .sort_values(["Missing Data Count", "Percentage Missing (%)"],
                 ascending=[False, False])
    .reset_index(drop=True)
)


# =========================
# Styling Helper
# =========================
def row_style(row):
    """Return full-row background style based on missingness.

    Dark red if any missing values, dark green if complete.

    Args:
        row: A pandas Series representing one row.

    Returns:
        List of inline CSS style strings (one per column).
    """
    if row["Missing Data Count"] > 0:
        return ["background-color:#8B0000; color:white; font-weight:bold;"] * len(row)
    return ["background-color:#006400; color:white; font-weight:bold;"] * len(row)


# =========================
# Styled Table Construction
# =========================
styled_missingness = (
    miss_df.style
    .format({"Percentage Missing (%)": "{:.2f}%"})
    .apply(row_style, axis=1)
    .set_properties(**{
        "border": "1px solid #444",
        "font-size": "12px",
        "color": "#e8e8e8",
    })
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("background-color", "#1f2c3b"),
                ("color", "#f1f1f1"),
                ("border", "1px solid #555"),
                ("font-weight", "bold"),
            ],
        },
        {
            "selector": "td",
            "props": [
                ("padding", "6px 8px"),
            ],
        },
    ])
)


# =========================
# Display DataFrame
# =========================
display(styled_missingness)

With reference to the output of the code cell above, we are able to confirm and identify that there are no missing values within the provided dataset. Thereforem we are able to procced to the next section where we will indentify and address any duplicated data within our dataset.

---
# 4. Identifying Duplicated Data

In this section, we will be identifying and addressing any duplicated data within our dataset. Duplicated data can cause multiple issues such as influencing the numerical summaries of our data and visualisation. Thereforem, should there be any duplicated data identified within our dataset, we will have to drop them.

In [ ]:
"""DAVI CA1 — Duplicate Row Analysis.

Checks for duplicate rows across all columns in the HDB Resale Flat Prices
dataset. A red flag table is displayed if duplicates exist; otherwise, a
green confirmation table is shown.
"""

# =========================
# Duplicate Count
# =========================
all_cols = df.columns.tolist()
dup_rows = int(df.duplicated(subset=all_cols, keep=False).sum())

summary = pd.DataFrame(
    [{"Metric": "Duplicated Rows", "Count": dup_rows}]
)


# =========================
# Styling Helpers
# =========================
def style_dark_red_flag(df_in):
    """Return a red-styled table for duplicate warnings.

    Args:
        df_in: DataFrame containing metric results.

    Returns:
        Styled DataFrame with dark red background and white text.
    """
    return (
        df_in.style
        .set_properties(**{
            "background-color": "#8B0000",
            "color": "white",
            "font-weight": "bold",
            "border": "1px solid #444",
            "font-size": "12px"
        })
        .set_table_styles([
            {
                "selector": "th",
                "props": [
                    ("background-color", "#1f2c3b"),
                    ("color", "#f1f1f1"),
                    ("border", "1px solid #555"),
                    ("font-weight", "bold")
                ]
            }
        ])
    )


def style_full_green(df_in):
    """Return a green-styled table when no duplicates are found.

    Args:
        df_in: DataFrame containing metric results.

    Returns:
        Styled DataFrame with dark green background and white text.
    """
    return (
        df_in.style
        .set_properties(**{
            "background-color": "#006400",
            "color": "white",
            "font-weight": "bold",
            "border": "1px solid #444",
            "font-size": "12px"
        })
        .set_table_styles([
            {
                "selector": "th",
                "props": [
                    ("background-color", "#1f2c3b"),
                    ("color", "#f1f1f1"),
                    ("border", "1px solid #555"),
                    ("font-weight", "bold")
                ]
            }
        ])
    )


# =========================
# Display Result
# =========================
if dup_rows > 0:
    display(style_dark_red_flag(summary))
else:
    display(style_full_green(summary))

With reference to the output of the code celL above, we are able to identify that there are a total of '606' duplicated rows within out dataset. Therefore, we will need to address the duplicated data by dropping these duplicated rows.

---
## 4.1 Addressing Duplicated Data

In this sub-section, we will be addressing the previously identified duplicated data and we will be dropping them. Subsequently, we will be re-evaluating to confirm that the data have been dropped successfully and we are able to proceed to the next step of the Data Cleaning.

In [ ]:
"""DAVI CA1 — Drop Exact Duplicates Across All Columns.

Removes all exact duplicate rows from the HDB Resale Flat Prices dataset,
then rechecks for remaining duplicates.
Displays a green flag if dataset is clean, or a red warning if duplicates remain.
"""

# =========================
# Drop Exact Duplicates
# =========================
before = len(df)
df = df.drop_duplicates(keep="first").reset_index(drop=True)
after = len(df)

# =========================
# Recheck for Remaining Duplicates
# =========================
dup_rows = int(df.duplicated(keep=False).sum())
summary = pd.DataFrame([
    {"Metric": "Duplicated Rows", "Count": dup_rows},
    {"Metric": "Rows Before", "Count": before},
    {"Metric": "Rows After", "Count": after},
    {"Metric": "Rows Removed", "Count": before - after},
])


# =========================
# Display Summary
# =========================
if dup_rows > 0:
    display(style_dark_red_flag(summary))
else:
    display(style_full_green(summary))

With reference to the code cell above, we are able to verify that the duplicated data have been successfully addressed and we are able to proceed to conduct inspection on the data type for the various features within the dataset.

---
# 5. Data Types Identification

In this section, we will be identifying the various Data Types that each of the Features of the dataset are in. This process will help us in identifying and addressing any data abbornalies later on. This is because with reference to the metadata, we will have a understanding of the intended data type of the feature.

In [ ]:
"""DAVI CA1 — Compact Dark-Styled Column Summary.

Generates a concise summary of all columns in the HDB Resale Flat Prices
dataset, displaying each column’s data type, number of unique values, and
an example entry.
Includes dark-theme styling with colour cues by dtype:
• Numeric → dark green
• Datetime → dark blue
• Boolean → purple
• Object / Category → dark grey
"""

# =========================
# Column Summary Table
# =========================
cols = list(df.columns)

summary = pd.DataFrame({
    "column": cols,
    "dtype": [df[c].dtype for c in cols],
    "n_unique": [df[c].nunique(dropna=True) for c in cols],
    "example_value": [
        df[c].dropna().iloc[0] if df[c].notna().any() else "" for c in cols
    ],
})


# =========================
# Styling Function
# =========================
def style_info_table(df_in):
    """Return a dark-styled summary table with dtype-based colouring.

    Args:
        df_in: DataFrame containing column metadata.

    Returns:
        Styled DataFrame with custom colour cues by dtype category.
    """
    sty = (
        df_in.style
        .set_properties(**{"border": "1px solid #333", "font-size": "12px"})
        .set_table_styles([
            {
                "selector": "th",
                "props": [
                    ("background-color", "#1f2c3b"),
                    ("color", "white"),
                    ("border", "1px solid #333"),
                    ("font-weight", "bold"),
                ],
            },
            {
                "selector": "td",
                "props": [("border", "1px solid #333")],
            },
        ])
        .format({"n_unique": "{:,}"})
    )


    # =========================
    # Dtype Colour Coding
    # =========================
    def colour_dtype(val):
        """Colour the dtype column according to data type."""
        t = str(val)
        if any(x in t for x in ["int", "float", "Int", "Float"]):
            return "background-color:#006400; color:white; font-weight:bold;"
        if "datetime64" in t or "datetime" in t:
            return "background-color:#0b3066; color:white; font-weight:bold;"
        if "bool" in t:
            return "background-color:#4b3f72; color:white; font-weight:bold;"
        return "background-color:#2b2b2b; color:#eaeaea;"

    sty = sty.map(colour_dtype, subset=pd.IndexSlice[:, ["dtype"]])


    # =========================
    # Cell-Specific Styling
    # =========================
    sty = sty.set_properties(
        subset=["example_value"],
        **{"background-color": "#262626", "color": "#eaeaea"},
    )
    sty = sty.set_properties(
        subset=["column"],
        **{"background-color": "#1e1e1e", "color": "#eaeaea", "font-weight": "bold"},
    )

    return sty


# =========================
# Display Styled Summary
# =========================
display(style_info_table(summary))

With reference to the output of the code cell above, we are able to verify the data types of the various features within the dataset. This subsequently tells us that there are data inconsistencies in data columns such as 'flat_type' which will need to be addressed in the next section.

---
# 6. Data Abnormalies and Inconsistencies

In this section, we will be identifying and addressing the various data abnormalieis and inconsistencies within the HDB dataset provided. These inspections will be conducted at the feature level and subsequently, should there be any data issues identified, we will provide an action plan so as to maintain the data integrity and clean the data with respect to the metadata provided and / or the unique data within that feature.

---
## 6.1 Abnormalies Identification in 'Month' Data Column

In this sub-section, we will be indentifying if there is any Data Abnormalies within the 'month' data column. Should there be any abnormalies, we will need to subsequently address the issues identified in another code cell. The strategy that we will utilise to identify abnormal data is summing each unique value of the data and sorting them in ascending order. This will allow us to look at 'rarer' values which will most likely be abnormalies as they are often one-off incidents.

In [ ]:
"""DAVI CA1 — Value Distribution (Counts) with Red Flags.

Builds a frequency table for a chosen column (here: 'month') and styles it
for dark mode. Uses a custom deep red gradient where rarer categories appear
darker red, and higher counts remain readable in muted tones — never white.
"""

# =========================
# Styling Helper
# =========================
def style_count_red_flag(df_in: pd.DataFrame, count_col: str = "count"):
    """Return a dark-themed table with a custom dark red gradient on counts.

    The lower the count, the darker the red, to flag rare categories.
    The higher the count, the lighter (but still dark-friendly) shade.

    Args:
        df_in: DataFrame containing the value counts.
        count_col: Name of the numeric column to colour by (default: "count").

    Returns:
        A styled DataFrame suitable for notebook display.
    """
    # Validate column and coerce numeric
    if count_col not in df_in.columns:
        raise KeyError(f"'{count_col}' not found in DataFrame columns: {list(df_in.columns)}")
    df_num = df_in.copy()
    df_num[count_col] = pd.to_numeric(df_num[count_col], errors="coerce").fillna(0)

    # Custom dark red colormap (deep maroon → medium red)
    dark_red_cmap = mpl.colors.LinearSegmentedColormap.from_list(
        "dark_red_cmap", ["#330000", "#660000", "#8B0000", "#AA1C1C", "#CC3333"]
    )

    sty = (
        df_num.style
        .background_gradient(cmap=dark_red_cmap, subset=[count_col], axis=None)
        .set_properties(**{
            "border": "1px solid #333",
            "font-size": "12px",
            "color": "#f0f0f0",
            "font-weight": "bold",
        })
        .set_table_styles([
            {
                "selector": "th",
                "props": [
                    ("background-color", "#1f2c3b"),
                    ("color", "white"),
                    ("border", "1px solid #333"),
                    ("font-weight", "bold"),
                ],
            },
            {
                "selector": "td",
                "props": [("border", "1px solid #333")],
            },
        ])
        .format({count_col: "{:,}"})
    )
    return sty


# =========================
# Inspect Unique Month Values
# =========================
if "month" in df.columns:
    month_df = (
        df["month"]
        .value_counts(dropna=False)
        .rename_axis("month")
        .reset_index(name="count")
        .sort_values("count", ascending=True, ignore_index=True)
    )

    display(style_count_red_flag(month_df, count_col="count"))
else:
    print("Column 'month' not found in DataFrame.")

With reference to the output of the code cell above, we are able to make the following conclusions and observatios on the data abnormalies.

- There are 2 Data Abnormalies observed

**Action Plan:**
1. Change '2024-02, woodlands' to '2024-02'

2. Change '2023-0113' to '2023-01'

**Justification**
1. Most likely a data entry error where there was an accidental insert of town due to the 2 features' close proximity.

2. Most likely a data entry error where the day of the month was included. Therefore, with reference to the other data entry, we will normalise to the month level.

With the action plan and justification indicated above, we will proceed to address the data abnormalies.

---
### 6.1.1 Abnormalies Addressment in 'Month' Data Column

In this sub-sub section, we will be addressing the data abnormalies we previously identified. We will carry out the action plan we indicated and subsequently verify if the changes have been applied to the Data Column.

In [ ]:
"""DAVI CA1 — Fix Month Column and Display Distribution.

Quickly corrects specific malformed entries in the 'month' column, then
display a frequency table with a green gradient to
highlight rarer categories.
"""

# =========================
# Value Fix
# =========================
if "month" in df.columns:
    # Direct mapping of known malformed values
    fix_map = {
        "2024-02, woodlands": "2024-02",
        "2023-0113": "2023-01",
    }
    df["month"] = df["month"].replace(fix_map)
else:
    raise KeyError("Column 'month' not found in DataFrame.")


# =========================
# Build Value Counts
# =========================
month_df = (
    df["month"]
    .value_counts(dropna=False)
    .rename_axis("month")
    .reset_index(name="count")
    .sort_values("count", ascending=True, ignore_index=True)
)


# =========================
# Display
# =========================
display(style_full_green(month_df))

With reference to the output of the code cell above, we are able to verify that the action plan previously mentioned have been successfully carried out and we are able to proceed to indentify and address other data abnormalies in the other features.

---
## 6.2 Abnormalies Identification in 'Town' Data Column

In this sub-section, we will be indentifying if there is any Data Abnormalies within the 'town' data column. Should there be any abnormalies, we will need to subsequently address the issues identified in another code cell. The strategy that we will utilise to identify abnormal data is summing each unique value of the data and sorting them in ascending order. This will allow us to look at 'rarer' values which will most likely be abnormalies as they are often one-off incidents.

In [ ]:
"""DAVI CA1 — Value Distribution (Counts) with Red Flags.

Builds a frequency table for a chosen column (here: 'town') and styles it
for dark mode. Low-frequency categories are highlighted with a reversed
red gradient (darker red = rarer), enabling quick anomaly/rarity scans.
"""

# =========================
# Inspect Unique Town Values
# =========================
if "town" in df.columns:
    town_df = (
        df["town"]
        .value_counts(dropna=False)
        .rename_axis("town")
        .reset_index(name="count")
        .sort_values("count", ascending=True, ignore_index=True)
    )
    display(style_count_red_flag(town_df, count_col="count"))
else:
    print("Column 'town' not found in DataFrame.")

With reference to the output of the code cell above, we are able to make the following conclusions and observatios on the data abnormalies.

- There are 6 Data Abnormalies observed

**Action Plan:**
1. Change 'ANG MO KIOO' to 'ANG MO KIO'

2. Change 'CCK' to 'CHOA CHU KANG'

3. Change 'SENG-KANG' to 'SENGKANG'

4. Change 'AMK' to 'ANG MO KIO'

5. Change 'SENGKANG North' to 'SENGKANG'

6. Change 'TAMPINES Park' to 'TAMPINES'

7. Standardise all Data Entry Format to First-Letter Capitalisation (i.e., 'ANG MO KIO' to 'Ang Mo Kio')


**Justification**
1. Most likely a data entry error where an additional 'O' is added.

2. Short form for Choa Chu Kang. Thus, standardisation to population.

3. Most likely a data entry error where a dash was added.

4. Short form for Ang Mo Kio. Thus, standardisation to population.

5. Despite 'SENGKANG North' potentially being a legitimate location, the data size is too small for any legitimate analysis to be conducted. Therefore, it should be merged with 'SENGKANG' as per the scope of the data.

6. Despite 'TAMPINES Park' potentially being a legitimate location, the data size is too small for any legitimate analysis to be conducted. Therefore, it should be merged with 'TAMPINES' as per the scope of the data.

7. By standardising all data format, data is able to be viewed easier on visualisation axis.


With the action plan and justification indicated above, we will proceed to address the data abnormalies.

---
### 6.2.1 Abnormalies Addressment in 'Town' Data Column

In this sub-sub section, we will be addressing the data abnormalies we previously identified. We will carry out the action plan we indicated and subsequently verify if the changes have been applied to the Data Column.

In [ ]:
"""DAVI CA1 — Fix Town Column and Display Distribution.

Quickly corrects specific malformed entries in the 'town' column, then
display a frequency table with a green gradient to
highlight rarer categories.
"""

# =========================
# Normalise and Map Variants
# =========================
if "town" not in df.columns:
    raise KeyError("Column 'town' not found in DataFrame.")

# Known inconsistent/short forms to canonical names
town_fix = {
    "ANG MO KIOO": "Ang Mo Kio",
    "AMK": "Ang Mo Kio",
    "CCK": "Choa Chu Kang",
    "SENG-KANG": "Sengkang",
    "SENGKANG NORTH": "Sengkang",
    "TAMPINES PARK": "Tampines",
}

# Normalise whitespace/case, apply mapping, then convert to Proper Case
df["town"] = (
    df["town"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.upper()
    .replace(town_fix)
    .str.title()
)

# =========================
# Verify via Display
# =========================
town_df_cleaned = (
    df["town"]
    .value_counts(dropna=False)
    .rename_axis("town")
    .reset_index(name="count")
    .sort_values("count", ascending=True, ignore_index=True)
)

# =========================
# Display DataFrame
# =========================
display(style_full_green(town_df_cleaned))

With reference to the output of the code cell above, we are able to verify that the action plan previously mentioned have been successfully carried out and we are able to proceed to indentify and address other data abnormalies in the other features.

---
## 6.3 Abnormalies Identification in 'flat_type' Data Column

In this sub-section, we will be indentifying if there is any Data Abnormalies within the 'flat_type' data column. Should there be any abnormalies, we will need to subsequently address the issues identified in another code cell. The strategy that we will utilise to identify abnormal data is summing each unique value of the data and sorting them in ascending order. This will allow us to look at 'rarer' values which will most likely be abnormalies as they are often one-off incidents.

In [ ]:
"""DAVI CA1 — Inspect Unique 'flat_type' Values (Red Flags for Low Counts).

Generates a frequency table of all unique flat types and applies the
existing dark red flag style to highlight rarer categories.
"""

# =========================
# Inspect Unique Flat Types
# =========================
if "flat_type" in df.columns:
    flat_type_df = (
        df["flat_type"]
        .value_counts(dropna=False)
        .rename_axis("flat_type")
        .reset_index(name="count")
        .sort_values("count", ascending=True, ignore_index=True)
    )

    display(style_count_red_flag(flat_type_df))
else:
    print("Column 'flat_type' not found in DataFrame.")

With reference to the output of the code cell above, we are able to make the following conclusions and observatios on the data abnormalies.

- There are 10 Data Abnormalies observed

**Action Plan:**
1. Change '3 ROOMS' to '3 ROOM'

2. Change '3 ROOMs' to '3 ROOM'

3. Change '4 ROOMSS' to '4 ROOM'

4. Change '4 ROOMS' to '4 ROOM'

5. Change '3-ROOM' to '3 ROOM'

6. Change '3 ROOMATES' to '3 ROOM'

7. Change '5-RM' to '5 ROOM'

8. Change '44 ROOM' to '4 ROOM'

9. Change 'EXECUTIVE part 1' to 'EXECUTIVE'

10. Change '3 RM' to '3 ROOM'

11. Standardise all Data Entry Format to First-Letter Capitalisation (i.e., '1 ROOM' to '1 Room')


**Justification**
1. Most likely a data entry error where a 'S' is added.

2. Most likely a data entry error where a 's' is added.

3. Most likely a data entry error where a 'SS' is added.

4. Most likely a data entry error where a 'S' is added.

5. Most likely a data entry error where a '-' is added.

6. Most likely a data entry error where 'ATES' is added.

7. Short form for 'ROOM'. Thus, standardisation to population.

8. Most likely a data entry error where an additional '4' was added.

9. Most likely a data entry error where 'part 1' is added.

10. Short form for 'ROOM'. Thus, standardisation to population.

11. By standardising all data format, data is able to be viewed easier on visualisation axis.


With the action plan and justification indicated above, we will proceed to address the data abnormalies.

---
### 6.3.1 Abnormalies Addressment in 'flat_type' Data Column

In this sub-sub section, we will be addressing the data abnormalies we previously identified. We will carry out the action plan we indicated and subsequently verify if the changes have been applied to the Data Column.

In [ ]:
"""DAVI CA1 — Clean & Standardise 'flat_type' Values (Normalised to 'Room').

Normalises the 'flat_type' column by fixing common typos and inconsistent
formats (e.g., '3 ROOMs' → '3 ROOM', '5-RM' → '5 ROOM'), ensuring a
uniform naming convention. Converts all cleaned entries to Title Case
for presentation or Tableau use.
"""

# =========================
# Clean & Normalise
# =========================
if "flat_type" in df.columns:
    # Mapping for known inconsistent / misspelled values
    flat_type_fix = {
        "1 ROOM": "1 ROOM",
        "2 ROOMS": "2 ROOM",
        "3 ROOMS": "3 ROOM",
        "3 ROOMs": "3 ROOM",
        "3-ROOM": "3 ROOM",
        "3 ROOMATES": "3 ROOM",
        "3 RM": "3 ROOM",
        "4 ROOMSS": "4 ROOM",
        "4 ROOMS": "4 ROOM",
        "44 ROOM": "4 ROOM",
        "5-RM": "5 ROOM",
        "EXECUTIVE PART 1": "EXECUTIVE",
        "MULTI GENERATION": "MULTI-GENERATION",
    }

    # Normalise spacing, case, and apply mapping
    df["flat_type"] = (
        df["flat_type"]
        .astype(str)
        .str.strip()
        .str.upper()
        .str.replace(r"\s+", " ", regex=True)
        .replace(flat_type_fix)
        .str.title()
    )

    # =========================
    # Verify Cleaned Unique Values
    # =========================
    flat_type_df_cleaned = (
        df["flat_type"]
        .value_counts(dropna=False)
        .rename_axis("flat_type")
        .reset_index(name="count")
        .sort_values("count", ascending=True, ignore_index=True)
    )

    display(style_full_green(flat_type_df_cleaned))
else:
    print("Column 'flat_type' not found in DataFrame.")

With reference to the output of the code cell above, we are able to verify that the action plan previously mentioned have been successfully carried out and we are able to proceed to indentify and address other data abnormalies in the other features.

---
## 6.4 Abnormalies Identification in 'Block' Data Column

In this sub-section, we will be indentifying if there is any Data Abnormalies within the 'block' data column. Should there be any abnormalies, we will need to subsequently address the issues identified in another code cell. The strategy that we will utilise to identify abnormal data is summing each unique value of the data and sorting them in ascending order. This will allow us to look at 'rarer' values which will most likely be abnormalies as they are often one-off incidents.

In [ ]:
"""DAVI CA1 — Inspect Unique 'block' Values (Red Flags for Low Counts).

Generates a frequency table of all unique block identifiers in the dataset
and applies the existing dark red flag style to highlight rarer or
underrepresented entries.
"""

# =========================
# Inspect Unique Block Values
# =========================
if "block" in df.columns:
    block_df = (
        df["block"]
        .value_counts(dropna=False)
        .rename_axis("block")
        .reset_index(name="count")
        .sort_values("count", ascending=True, ignore_index=True)
    )

    display(style_count_red_flag(block_df))
else:
    print("Column 'block' not found in DataFrame.")

With reference to the output of the code cell above, we are able to make the following conclusions and observatios on the data abnormalies.

- There are Multiple Data Abnormalies observed

**Action Plan:**
1. Remove all Special Characters

2. Remove '_BLK'


**Justification**
1. Most likely a data entry error where special characters are added.

2. Most likely a data entry error where '_BLK' is added.

With the action plan and justification indicated above, we will proceed to address the data abnormalies.

---
### 6.4.1 Abnormalies Addressment in 'Block' Data Column

In this sub-sub section, we will be addressing the data abnormalies we previously identified. We will carry out the action plan we indicated and subsequently verify if the changes have been applied to the Data Column.

In [ ]:
"""DAVI CA1 — Clean & Standardise 'block' Values + Audit & Verify.

Normalises HDB block identifiers (e.g., removes 'BLK', strips non-alphanumerics,
uppercases, and trims leading zeros), then audits changes and verifies the final
distribution. Uses existing dark-mode stylers:
    • style_dark_red_flag(...) for change mappings (rarer → darker red)
    • style_full_green(...) for final frequency table
"""

# =========================
# Guard
# =========================
if "block" not in df.columns:
    raise KeyError("Column 'block' not found in DataFrame.")


# =========================
# Cleaner
# =========================
def clean_block_value(x: Optional[object]) -> Optional[str]:
    """Normalise an HDB block label into a compact, canonical token.

    Rules:
        • Remove 'BLK' or '_BLK' tokens anywhere.
        • Strip all non-alphanumeric characters.
        • Uppercase and trim.
        • Remove leading zeros from numeric prefixes (e.g., '0012' → '12').
        • Return None if nothing meaningful remains.

    Args:
        x: Raw block value.

    Returns:
        A cleaned block string or None if nothing valid remains.
    """
    if pd.isna(x):
        return None

    s = str(x).upper().strip()
    s = re.sub(r"\s+", " ", s)

    # Remove standalone / joined BLK tokens
    s = re.sub(r"(^|\W)BLK($|\W)", " ", s)  # standalone BLK
    s = s.replace("_BLK", " ").replace("BLK_", " ")
    s = s.strip()

    # Keep only A–Z and 0–9
    s = re.sub(r"[^A-Z0-9]", "", s)
    if not s:
        return None

    # Strip leading zeros if starts with digits
    m = re.match(r"^0+(\d.*)$", s)
    if m:
        s = m.group(1)

    return s or None


# =========================
# Apply Cleaning
# =========================
block_before = df["block"].copy()
df["block"] = df["block"].map(clean_block_value)

# =========================
# Audit
# =========================
audit = pd.DataFrame({"original": block_before, "cleaned": df["block"]})

changed = audit.loc[
    audit["original"].notna()
    & audit["cleaned"].notna()
    & (audit["original"].astype(str).str.upper() != audit["cleaned"])
]

mapping_counts = (
    changed.groupby(["original", "cleaned"], dropna=False)
    .size()
    .reset_index(name="count_changes")
    .sort_values(["count_changes", "original"], ascending=[False, True], ignore_index=True)
)

print(f"Changed mappings: {len(mapping_counts)}")

mc_view = mapping_counts.rename(columns={"count_changes": "count"})
display(style_dark_red_flag(mc_view.head(50)))

# =========================
# Verify
# =========================
block_freq = (
    df["block"]
    .value_counts(dropna=False)
    .rename_axis("block")
    .reset_index(name="count")
    .sort_values("count", ascending=False, ignore_index=True)
)

# =========================
# Display DataFrame
# =========================
display(style_full_green(block_freq))

With reference to the output of the code cell above, we are able to verify that the action plan previously mentioned have been successfully carried out and we are able to proceed to indentify and address other data abnormalies in the other features.

---
## 6.5 Abnormalies Identification in 'Street_name' Data Column

In this sub-section, we will be indentifying if there is any Data Abnormalies within the 'street_name' data column. Should there be any abnormalies, we will need to subsequently address the issues identified in another code cell. The strategy that we will utilise to identify abnormal data is summing each unique value of the data and sorting them in ascending order. This will allow us to look at 'rarer' values which will most likely be abnormalies as they are often one-off incidents.

In [ ]:
"""DAVI CA1 — Inspect Unique 'street_name' Values (Red Flags for Low Counts).

Generates a frequency table of all unique street names in the dataset and applies
the existing dark red flag style to highlight streets with very few occurrences.
This helps surface rare or potentially erroneous entries.
"""

# =========================
# Inspect Unique Street Names
# =========================
if "street_name" in df.columns:
    street_name_df = (
        df["street_name"]
        .value_counts(dropna=False)
        .rename_axis("street_name")
        .reset_index(name="count")
        .sort_values("count", ascending=True, ignore_index=True)
    )

    display(style_count_red_flag(street_name_df))
else:
    print("Column 'street_name' not found in DataFrame.")


With reference to the output of the code cell above, we are able to make the following conclusions and observatios on the data abnormalies.

- There are Multiple Data Abnormalies observed

**Action Plan:**
1. Standardise all Data Entry Format to Full Term (i.e., 'AVE' to 'Avenue')

2. Standardise all Data Entry Format to First-Letter Capitalisation (i.e., '1 ROOM' to '1 Room')


**Justification**
1. Short forms should be standardised so that geospatial tools can easily recognise location.

2. By standardising all data format, data is able to be viewed easier on visualisation axis.


With the action plan and justification indicated above, we will proceed to address the data abnormalies.

---
### 6.5.1 Abnormalies Addressment in 'Street_name' Data Column

In this sub-sub section, we will be addressing the data abnormalies we previously identified. We will carry out the action plan we indicated and subsequently verify if the changes have been applied to the Data Column.

In [ ]:
"""DAVI CA1 — Address Normalisation.

Performs full standardisation of Singapore HDB addresses using existing
columns: 'block' and 'street_name'.

Fixes bug where valid road-type suffixes (e.g., 'RING RD') were truncated.
Displays audit summary in solid green.
"""

# =========================
# Guards
# =========================
_required = [c for c in ["block", "street_name"] if c not in df.columns]
if _required:
    raise KeyError(f"Missing required columns: {_required}")


# =========================
# Block Normaliser
# =========================
def _clean_block_value(x: str | None) -> str | None:
    """Return canonical block token (no 'BLK', only alphanumerics, no leading zeros)."""
    if pd.isna(x):
        return None
    s = str(x).upper().strip()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"(^|[^A-Z0-9])BLK([^A-Z0-9]|$)", " ", s)
    s = re.sub(r"[^A-Z0-9]", "", s)
    if not s:
        return None
    m = re.match(r"^0+(\d.*)$", s)
    if m:
        s = m.group(1)
    return s


# =========================
# Street Normaliser
# =========================
_PRE_REPLACERS = [
    (r"\bCTeRL\b", "CENTRAL"),
    (r"\bCeRL\b", "CENTRAL"),
    (r"\bCL\s+CL\b", "CL"),
    (r"\bLINK\s+LINK\b", "LINK"),
    (r"\bWAY\s+LINK\b", "WAY"),
    (r"\bSPORE\b", ""),  # noise
]

_TOKEN_EXPAND = {
    "BT": "BUKIT",
    "JLN": "JALAN",
    "LOR": "LORONG",
    "UPP": "UPPER",
    "KG": "KAMPONG",
    "TP": "TOA PAYOH",
    "TPY": "TOA PAYOH",
    "AMK": "ANG MO KIO",
    "C'WEALTH": "COMMONWEALTH",
    "NTH": "NORTH",
    "STH": "SOUTH",
    "CTR": "CENTRAL",
    "CTRL": "CENTRAL",
    "PK": "PARK",
    "GDNS": "GARDENS",
    "HTS": "HEIGHTS",
    "MKT": "MARKET",
}

_ROAD_TYPES_MAP = {
    "RD": "ROAD",
    "AVE": "AVENUE",
    "ST": "STREET",
    "DR": "DRIVE",
    "CRES": "CRESCENT",
    "CL": "CLOSE",
    "PL": "PLACE",
    "TER": "TERRACE",
    "PK": "PARK",
    "HTS": "HEIGHTS",
    "BOW": "BOW",
    "LINK": "LINK",
    "LANE": "LANE",
    "LOOP": "LOOP",
    "RING": "RING",
    "CIRCLE": "CIRCLE",
    "RISE": "RISE",
    "VIEW": "VIEW",
    "WALK": "WALK",
    "CTRL": "CENTRAL",
}
_ROAD_TYPES = set(_ROAD_TYPES_MAP) | set(_ROAD_TYPES_MAP.values())


def _collapse_dupes(tokens: list[str]) -> list[str]:
    """Collapse immediate duplicate tokens (e.g., 'ROAD ROAD' → 'ROAD')."""
    out = []
    for t in tokens:
        if not out or out[-1] != t:
            out.append(t)
    return out


def _collapse_adjacent_roadtypes(tokens: list[str]) -> list[str]:
    """Collapse consecutive road-type tokens except final one (keep last type)."""
    out = []
    for i, t in enumerate(tokens):
        if out and out[-1] in _ROAD_TYPES and t in _ROAD_TYPES and i < len(tokens) - 1:
            continue
        out.append(t)
    return out


def _clean_street_geocode(x: str | None) -> str | None:
    """Return a geocoder/Tableau-friendly street name with full words and Title Case."""
    if pd.isna(x):
        return None
    s = str(x).strip().upper()
    for pat, repl in _PRE_REPLACERS:
        s = re.sub(pat, repl, s)
    s = re.sub(r"[^A-Z0-9\s'\-\.]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    if not s:
        return None

    exp = []
    for t in s.split():
        t = _TOKEN_EXPAND.get(t, t)
        t = _TOKEN_EXPAND.get(t, t)
        exp.extend(t.split())

    norm = [_ROAD_TYPES_MAP.get(t, t) for t in exp]
    norm = [t for t in norm if t != "SPORE"]
    norm = _collapse_dupes(norm)
    norm = _collapse_adjacent_roadtypes(norm)
    if not norm:
        return None

    pretty = [w if w.isdigit() else w.title() for w in norm]
    return " ".join(pretty)


# =========================
# Preserve Originals for Audit
# =========================
df["_block_orig"] = df["block"]
df["_street_orig"] = df["street_name"]

# =========================
# Apply In Place
# =========================
df["block"] = df["block"].astype(object).map(_clean_block_value)
df["street_name"] = df["street_name"].astype(object).map(_clean_street_geocode)

# =========================
# Audit & Verify
# =========================
_changes = (
    pd.DataFrame({"original": df["_street_orig"], "cleaned": df["street_name"]})
    .dropna(subset=["original", "cleaned"])
    .assign(
        o=lambda d: d["original"].astype(str).str.strip().str.upper(),
        c=lambda d: d["cleaned"].astype(str).str.strip().str.upper(),
    )
    .query("o != c")
    .groupby(["original", "cleaned"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(["count", "original"], ascending=[False, True])
    .head(50)
)

print(f"Street mappings changed: {len(_changes)}")

if not _changes.empty:
    audit_view = _changes[["original", "cleaned", "count"]]
    display(style_full_green(audit_view))
else:
    print("No street name changes detected.")

# =========================
# Preview
# =========================
_preview_cols = ["_street_orig", "street_name"]
display(style_full_green(df.loc[:, _preview_cols].head(15)))

# =========================
# Cleanup
# =========================
df.drop(columns=["_block_orig", "_street_orig"], inplace=True, errors="ignore")

With reference to the output of the code cell above, we are able to verify that the action plan previously mentioned have been successfully carried out and we are able to proceed to indentify and address other data abnormalies in the other features.

---
## 6.6 Abnormalies Identification in 'Storey_range' Data Column

In this sub-section, we will be indentifying if there is any Data Abnormalies within the 'storey_range' data column. Should there be any abnormalies, we will need to subsequently address the issues identified in another code cell. The strategy that we will utilise to identify abnormal data is summing each unique value of the data and sorting them in ascending order. This will allow us to look at 'rarer' values which will most likely be abnormalies as they are often one-off incidents.

In [ ]:
"""DAVI CA1 — Inspect Unique 'storey_range' Values (Red Flags for Low Counts).

Generates a frequency table of all unique storey ranges in the dataset
and applies the existing dark red flag style to highlight rarer or
less common storey ranges for quality assurance.
"""

# =========================
# Inspect Unique Storey Ranges
# =========================
if "storey_range" in df.columns:
    storey_range_df = (
        df["storey_range"]
        .value_counts(dropna=False)
        .rename_axis("storey_range")
        .reset_index(name="count")
        .sort_values("count", ascending=True, ignore_index=True)
    )

    display(style_count_red_flag(storey_range_df))
else:
    print("Column 'storey_range' not found in DataFrame.")

With reference to the output of the code cell above, we are able to make the following conclusions and observatios on the data abnormalies.

- There are 20 Data Abnormalies observed

**Action Plan:**
1. Change '01@@03' to '01 TO 03'

2. Change '07@@09' to '07 TO 09'

3. Change '10--12' to '10 TO 12'

4. Change '07 --- 09' to '07 TO 09'

5. Change '01 ---03' to '01 TO 03'

6. Change '13 - 15' to '13 TO 15'

7. Change '10-Dec' to '10 TO 12'

8. Change '01----03' to '01 TO 03'

9. Change '04---06' to '04 TO 06'

10. Change '01@03' to '01 TO 03'

11. Change '10@12' to '10 TO 12'

12. Change '04 TO 06-10' to '04 TO 06'

13. Change '13 TO 15-9' to '13 TO 15'

14. Change '01 -- 03' to '01 TO 03'

15. Change '22 -- 24' to '22 TO 24'

16. Change '22 --- 24' to '22 TO 24'

17. Change '4-Jun' to '04 TO 06'

18. Change '07 -- 09' to '07 TO 09'

19. Change '19 -- 21' to '19 TO 21'

20. Change '04 -- 06' to '04 TO 06'

21. Standardise all Data Entry Format to No Capitalisation (i.e., '04 TO 06' to '04 to 06')


**Justification**
1. Most likely a data entry error where '@@' was added instead of 'TO'.

2. Most likely a data entry error where '@@' was added instead of 'TO'.

3. Most likely a data entry error where '--' was added instead of 'TO'.

4. Most likely a data entry error where ' --- ' was added instead of 'TO'.

5. Most likely a data entry error where ' ---' was added instead of 'TO'.

6. Most likely a data entry error where '-' was added instead of 'TO'.

7. Most likely data entry software perceived '12' as 'December' due to similarity to Datatime format.

8. Most likely a data entry error where '----' was added instead of 'TO'.

9. Most likely a data entry error where '---' was added instead of 'TO'.

10. Most likely a data entry error where '@' was added instead of 'TO'.

11. Most likely a data entry error where '@' was added instead of 'TO'.

12. Most likely data entry software perceived entry as date due to similarity to Datatime format.

13. Most likely data entry software perceived entry as date due to similarity to Datatime format.

14. Most likely a data entry error where '--' was added instead of 'TO'.

15. Most likely a data entry error where '--' was added instead of 'TO'.

16. Most likely a data entry error where '---' was added instead of 'TO'.

17. Most likely data entry software perceived '6' as 'June' due to similarity to Datatime format.

18. Most likely a data entry error where '--' was added instead of 'TO'.

19. Most likely a data entry error where '--' was added instead of 'TO'.

20. Most likely a data entry error where '--' was added instead of 'TO'.

21. By standardising all data format, data is able to be viewed easier on visualisation axis.


With the action plan and justification indicated above, we will proceed to address the data abnormalies.

---
### 6.6.1 Abnormalies Addressment in 'Storey_range' Data Column

In this sub-sub section, we will be addressing the data abnormalies we previously identified. We will carry out the action plan we indicated and subsequently verify if the changes have been applied to the Data Column.

In [ ]:
"""DAVI CA1 — Clean & Standardise 'storey_range' Values (In-Place).

Normalises storey range data to a consistent 'N to N' format.
Handles common junk entries, replaces inconsistent separators,
and ensures all values follow a clean, human-readable structure.
Displays final cleaned unique values in solid dark green.
"""

# =========================
# Manual Fix Mapping for Junk Entries
# =========================
storey_map = {
    "01@@03": "01 TO 03",
    "07@@09": "07 TO 09",
    "10--12": "10 TO 12",
    "07 --- 09": "07 TO 09",
    "01 ---03": "01 TO 03",
    "13 - 15": "13 TO 15",
    "10-DEC": "10 TO 12",
    "01----03": "01 TO 03",
    "04---06": "04 TO 06",
    "01@03": "01 TO 03",
    "10@12": "10 TO 12",
    "04 TO 06-10": "04 TO 06",
    "13 TO 15-9": "13 TO 15",
    "01 -- 03": "01 TO 03",
    "22 -- 24": "22 TO 24",
    "22 --- 24": "22 TO 24",
    "4-JUN": "04 TO 06",
    "07 -- 09": "07 TO 09",
    "19 -- 21": "19 TO 21",
    "04 -- 06": "04 TO 06",
}


# =========================
# Clean & Normalise
# =========================
df["storey_range"] = (
    df["storey_range"]
    .astype(str)
    .str.strip()
    .str.upper()
    .replace(storey_map)
)


# =========================
# Standardise Format
# =========================
def normalise_storey_text(s: str) -> str:
    """Convert 'NN TO NN' → 'N to N' (lowercase 'to', no leading zeros)."""
    if pd.isna(s):
        return None
    s = s.strip().upper()
    if not re.fullmatch(r"\d{1,2}\s*TO\s*\d{1,2}", s):
        return s  # leave unexpected patterns as-is
    lo, hi = re.findall(r"\d{1,2}", s)
    lo, hi = int(lo), int(hi)
    return f"{lo} to {hi}"


df["storey_range"] = df["storey_range"].map(normalise_storey_text)
df["storey_range"] = df["storey_range"].replace({"": None, "NAN": None})


# =========================
# Inspect Cleaned Unique Values
# =========================
storey_cleaned = (
    df["storey_range"]
    .value_counts(dropna=False)
    .rename_axis("storey_range")
    .reset_index(name="count")
    .sort_values("count", ascending=True, ignore_index=True)
)


# =========================
# Display DataFrame
# =========================
display(style_full_green(storey_cleaned))

With reference to the output of the code cell above, we are able to verify that the action plan previously mentioned have been successfully carried out and we are able to proceed to indentify and address other data abnormalies in the other features.

---
## 6.7 Abnormalies Identification in 'Flat_model' Data Column

In this sub-section, we will be indentifying if there is any Data Abnormalies within the 'flat_model' data column. Should there be any abnormalies, we will need to subsequently address the issues identified in another code cell. The strategy that we will utilise to identify abnormal data is summing each unique value of the data and sorting them in ascending order. This will allow us to look at 'rarer' values which will most likely be abnormalies as they are often one-off incidents.

In [ ]:
"""DAVI CA1 — Inspect Unique 'flat_model' Values (Red Flags for Low Counts).

Generates a frequency table of all unique flat models in the dataset
and applies the existing dark red flag style to highlight rarer models.
This helps identify uncommon or legacy HDB flat types.
"""

# =========================
# Inspect Unique Flat Models
# =========================
if "flat_model" in df.columns:
    flat_model_df = (
        df["flat_model"]
        .value_counts(dropna=False)
        .rename_axis("flat_model")
        .reset_index(name="count")
        .sort_values("count", ascending=True, ignore_index=True)
    )

    display(style_count_red_flag(flat_model_df))
else:
    print("Column 'flat_model' not found in DataFrame.")


With reference to the output of the code cell above, we are able to make the following conclusions and observatios on the data abnormalies.

- There are 5 Data Abnormalies observed

**Action Plan:**
1. Change 'NG' to 'New Generation'

2. Change 'Improvedv1' to 'Improved'

3. Change 'Model -A' to 'Model A'

4. Change 'Model A_B' to 'Model A'

5. Change 'New-Generation' to 'New Generation'



**Justification**
1. 'NG' is short form for 'New Generation'.

2. Most likely a data entry error where ' v1 was added.

3. Most likely a data entry error where '' was added.

4. Most likely a data entry error where '_B' was added.

5. Most likely a data entry error where '-' was added.

With the action plan and justification indicated above, we will proceed to address the data abnormalies.

---
### 6.7.1 Abnormalies Addressment in 'Flat_model' Data Column

In this sub-sub section, we will be addressing the data abnormalies we previously identified. We will carry out the action plan we indicated and subsequently verify if the changes have been applied to the Data Column.

In [ ]:
"""DAVI CA1 — Clean & Normalise 'flat_model' Values (In-Place).

Standardises all 'flat_model' entries by mapping known variants,
fixing common typos, and aligning values to an approved canonical list.
Ensures consistent title-cased output (e.g., 'MODEL A' → 'Model A').
Displays final frequency table in solid dark green for verification.
"""

# =========================
# Guard
# =========================
if "flat_model" not in df.columns:
    raise KeyError("Column 'flat_model' not found in DataFrame.")


# =========================
# Canonical Reference Lists
# =========================
ACCEPTABLE = [
    "2-room", "3Gen", "Adjoined flat", "Apartment", "DBSS", "Improved",
    "Improved-Maisonette", "Maisonette", "Model A", "Model A2",
    "Model A-Maisonette", "Multi Generation", "New Generation",
    "Premium Apartment", "Premium Apartment Loft", "Premium Maisonette",
    "Simplified", "Standard", "Terrace", "Type S1", "Type S2"
]
canon_lut = {s.lower(): s for s in ACCEPTABLE}

# Known dirty to clean
DIRTY_MAP_UPPER = {
    "NG": "New Generation",
    "IMPROVEDV1": "Improved",
    "MODEL -A": "Model A",
    "MODEL A_B": "Model A",
    "NEW-GENERATION": "New Generation",
    "PREMIUM MAISONETTE": "Premium Maisonette",
    "IMPROVED-MAISONETTE": "Improved-Maisonette",
    "3GEN": "3Gen",
    "MULTI GENERATION": "Multi Generation",
    "TERRACE": "Terrace",
    "PREMIUM APARTMENT LOFT": "Premium Apartment Loft",
    "TYPE S2": "Type S2",
    "2-ROOM": "2-room",
    "TYPE S1": "Type S1",
    "ADJOINED FLAT": "Adjoined flat",
    "MODEL A-MAISONETTE": "Model A-Maisonette",
    "MODEL A2": "Model A2",
    "DBSS": "DBSS",
    "STANDARD": "Standard",
    "MAISONETTE": "Maisonette",
    "APARTMENT": "Apartment",
    "SIMPLIFIED": "Simplified",
    "PREMIUM APARTMENT": "Premium Apartment",
    "NEW GENERATION": "New Generation",
    "IMPROVED": "Improved",
    "MODEL A": "Model A",
}


# =========================
# Cleaning Function
# =========================
def normalise_flat_model_just_map(x: object) -> object:
    """Return canonicalised flat model using known mappings and light cleanup."""
    if pd.isna(x):
        return x

    s0 = str(x).strip()
    if not s0:
        return s0

    su = s0.upper()
    if su in DIRTY_MAP_UPPER:
        return DIRTY_MAP_UPPER[su]

    # Light normalisation (spaces, underscores, hyphens)
    s1 = s0.replace("_", " ")
    s1 = re.sub(r"\s+", " ", s1).strip()
    s1 = re.sub(r"-{2,}", "-", s1).strip()

    # Canonicalise (case-insensitive lookup)
    key = s1.lower()
    if key in canon_lut:
        return canon_lut[key]

    s2 = s1.title()
    if s2.lower() in canon_lut:
        return canon_lut[s2.lower()]

    # Fallback: return cleaned user input unchanged
    return s1


# =========================
# Apply Normalisation
# =========================
df["flat_model"] = df["flat_model"].map(normalise_flat_model_just_map)


# =========================
# Verify Cleaned Values
# =========================
flat_model_cleaned = (
    df["flat_model"]
    .value_counts(dropna=False)
    .rename_axis("flat_model")
    .reset_index(name="count")
    .sort_values("count", ascending=False, ignore_index=True)
)


# =========================
# Display DataFrame
# =========================
display(style_full_green(flat_model_cleaned))

With reference to the output of the code cell above, we are able to verify that the action plan previously mentioned have been successfully carried out and we are able to proceed to indentify and address other data abnormalies in the other features.

---
## 6.8 Abnormalies Identification in 'Remaining_lease' Data Column

In this sub-section, we will be indentifying if there is any Data Abnormalies within the 'remaining_lease' data column. Should there be any abnormalies, we will need to subsequently address the issues identified in another code cell. The strategy that we will utilise to identify abnormal data is summing each unique value of the data and sorting them in ascending order. This will allow us to look at 'rarer' values which will most likely be abnormalies as they are often one-off incidents.

In [ ]:
"""DAVI CA1 — Inspect Unique 'remaining_lease' Values (Red Flags for Low Counts).

Generates a frequency table of all unique 'remaining_lease' values in the dataset
and applies the existing dark red flag style to highlight rare or inconsistent
lease durations for further data validation.
"""

# =========================
# Inspect Unique Remaining Lease Values
# =========================
if "remaining_lease" in df.columns:
    remaining_lease_df = (
        df["remaining_lease"]
        .value_counts(dropna=False)
        .rename_axis("remaining_lease")
        .reset_index(name="count")
        .sort_values("count", ascending=True, ignore_index=True)
    )

    display(style_count_red_flag(remaining_lease_df))
else:
    print("Column 'remaining_lease' not found in DataFrame.")

With reference to the output of the code cell above, we are able to make the following conclusions and observatios on the data abnormalies.

- There are Multiple Data Abnormalies observed

**Action Plan:**
1. Change 'y' to 'years'

2. Change 'ys' to 'years'

3. Change data entry without months to '0 months'

4. Standardise all Data Entry Format to First-word Capitalisation (i.e., '42 years 10 months' to '42 Years 10 Months')



**Justification**
1. 'y' is short form for 'years'.

2. 'ys' is short form for 'years'.

3. To ensure data consistency during Feature Engineering.

3. By standardising all data format, data is able to be viewed easier on visualisation axis.

With the action plan and justification indicated above, we will proceed to address the data abnormalies.

---
### 6.8.1 Abnormalies Addressment in 'Remaining_lease' Data Column

In this sub-sub section, we will be addressing the data abnormalies we previously identified. We will carry out the action plan we indicated and subsequently verify if the changes have been applied to the Data Column.

In [ ]:
"""DAVI CA1 — Normalise 'remaining_lease' to 'X years Y months' (In-Place).

Cleans and standardises all 'remaining_lease' values into a fixed format:
'X years Y months' (always showing both parts, even when 0).
Also adds a helper numeric column 'remaining_lease_months'.

Steps:
    • L1) Token normalisation (handle shorthands/typos like 'yrs', 'mths', '10-12')
    • L2) Parse years/months (labelled or positional), roll months ≥ 12 into years
    • L3) Guard against negatives and None
    • L4) Format string consistently + create numeric months helper
    • L5) Verify with a dark-green frequency table
"""

# =========================
# Guard
# =========================
if "remaining_lease" not in df.columns:
    raise KeyError("Column 'remaining_lease' not found in DataFrame.")


# =========================
# Token Normalisation
# =========================
def _norm_tokens(s: object) -> str:
    """Return a cleaned, lowercased token string with unified units and separators."""
    t = str(s).strip().lower()
    # Normalise unit shorthands/typos
    t = re.sub(r"\bys?\b|\byr?s?\b", " years ", t)
    t = re.sub(r"\bmos?\b|\bmth?s?\b|\bmnths?\b", " months ", t)
    # Unify separators
    t = t.replace("/", " ").replace("-", " ").replace("_", " ")
    t = re.sub(r"[,\(\)]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t


# =========================
# Parse years/months
# =========================
def _parse_remaining_lease(val: object):
    """Parse 'remaining_lease' into (years, months) tuple or None if invalid."""
    if pd.isna(val):
        return None

    s = _norm_tokens(val)
    if not s:
        return None

    years = months = None

    # Labelled captures
    m = re.search(r"(\d+)\s*years?", s)
    if m:
        years = int(m.group(1))

    m = re.search(r"(\d+)\s*months?", s)
    if m:
        months = int(m.group(1))

    # Fallback: positional numbers (first to years, second to months)
    if years is None and months is None:
        nums = [int(n) for n in re.findall(r"\d+", s)]
        if len(nums) >= 1:
            years = nums[0]
        if len(nums) >= 2:
            months = nums[1]

    if years is None and months is None:
        return None

    years = int(years) if years is not None else 0
    months = int(months) if months is not None else 0

    # Roll months into years
    if months >= 12:
        years += months // 12
        months = months % 12

    # Guard against negatives
    years = max(0, years)
    months = max(0, months)

    return years, months


# =========================
# Format Fixed-string with Numeric Helper
# =========================
def _format_fixed(ym) -> str | None:
    """Return 'X years Y months' for a (years, months) tuple; None if missing."""
    if ym is None:
        return None
    y, m = ym
    return f"{y} years {m} months"


parsed = df["remaining_lease"].apply(_parse_remaining_lease)
df["remaining_lease"] = parsed.apply(_format_fixed)
df["remaining_lease_months"] = parsed.apply(
    lambda ym: None if ym is None else ym[0] * 12 + ym[1]
)


# =========================
# Verify
# =========================
lease_cleaned = (
    df["remaining_lease"]
    .value_counts(dropna=False)
    .rename_axis("remaining_lease")
    .reset_index(name="count")
    .sort_values("count", ascending=False, ignore_index=True)
)


# =========================
# Verify
# ========================
display(style_full_green(lease_cleaned))

With reference to the output of the code cell above, we are able to verify that the action plan previously mentioned have been successfully carried out and we are able to proceed to indentify and address other data abnormalies in the other features.

---
## 6.9 Abnormalies Identification in 'Resale_price' Data Column

In this sub-section, we will be indentifying if there is any Data Abnormalies within the 'resale_price' data column. Should there be any abnormalies, we will need to subsequently address the issues identified in another code cell. The strategy that we will utilise to identify abnormal data is summing each unique value of the data and sorting them in ascending order. This will allow us to look at 'rarer' values which will most likely be abnormalies as they are often one-off incidents.

In [ ]:
"""DAVI CA1 — Inspect Non-Numeric and Negative 'resale_price' Values (Red Flags).

Scans 'resale_price' for:
    • Non-numeric entries (strings, symbols, or mixed text)
    • Negative numeric values (invalid resale prices)

Displays each category in separate red-styled tables for anomaly validation.
"""

# =========================
# Guard
# =========================
if "resale_price" not in df.columns:
    raise KeyError("Column 'resale_price' not found in DataFrame.")


# =========================
# Detect Non-Numeric Entries
# =========================
resale_str = df["resale_price"].astype(str).str.strip()
is_numeric = pd.to_numeric(resale_str, errors="coerce").notna()
non_numeric = resale_str[~is_numeric]

# =========================
# Summarise Non-Numeric
# =========================
if non_numeric.empty:
    print("No non-numeric 'resale_price' entries found.")
else:
    non_numeric_df = (
        non_numeric.value_counts(dropna=False)
        .rename_axis("resale_price_raw")
        .reset_index(name="count")
        .sort_values("count", ascending=False, ignore_index=True)
    )
    print(f"Non-numeric resale_price entries found: {len(non_numeric_df):,}")
    display(style_dark_red_flag(non_numeric_df))


# =========================
# Detect Negative Values
# =========================
resale_numeric = pd.to_numeric(df["resale_price"], errors="coerce")
negatives_df = df.loc[resale_numeric < 0, ["town", "block", "flat_type", "resale_price"]]

# =========================
# Display Negative Values
# =========================
if negatives_df.empty:
    print("No negative resale_price values found.")
else:
    print(f"Negative resale_price values found: {len(negatives_df):,}")
    display(style_dark_red_flag(negatives_df.head(50)))

With reference to the output of the code cell above, we are able to make the following conclusions and observatios on the data abnormalies.

- There are Multiple Data Abnormalies observed

**Action Plan:**
1. Remove all '-'

**Justification**
1. Most likely due to data entry error where '-' was added.


With the action plan and justification indicated above, we will proceed to address the data abnormalies.

---
### 6.9.1 Abnormalies Addressment in 'Resale_price' Data Column

In this sub-sub section, we will be addressing the data abnormalies we previously identified. We will carry out the action plan we indicated and subsequently verify if the changes have been applied to the Data Column.

In [ ]:
"""DAVI CA1 — Clean and Convert 'resale_price' to Float (In-Place).

Cleans all 'resale_price' values by removing formatting noise (e.g., $, commas, dashes),
then converts them to float type for quantitative analysis. Also verifies conversion
success rate and displays a dark-green summary table.
"""

# =========================
# Guard
# =========================
if "resale_price" not in df.columns:
    raise KeyError("Column 'resale_price' not found in DataFrame.")


# =========================
# Cleaning Function
# =========================
def clean_resale_price(x: object) -> str | None:
    """Remove non-numeric symbols and minor formatting issues from price values."""
    if pd.isna(x):
        return None

    s = str(x).strip()
    # Remove common non-digit symbols
    s = re.sub(r"[\$,]", "", s)
    s = re.sub(r"-", "", s)
    s = s.replace(" ", "")
    return s


# =========================
# Clean and Convert
# =========================
df["resale_price"] = df["resale_price"].map(clean_resale_price)
df["resale_price"] = pd.to_numeric(df["resale_price"], errors="coerce")


# =========================
# Verify Conversion
# =========================
non_numeric_after = df["resale_price"].isna().sum()
total_rows = len(df)

print("resale_price successfully converted to float.")
print(f"Total rows: {total_rows:,}")
print(f"Non-numeric after cleaning: {non_numeric_after:,} ({non_numeric_after / total_rows:.2%})")


# =========================
# Summary Preview
# =========================
resale_summary = (
    df["resale_price"]
    .describe()
    .to_frame()
    .rename(columns={"resale_price": "value"})
)


# =========================
# Display DataFrame
# =========================
display(style_full_green(resale_summary))

With reference to the output of the code cell above, we are able to verify that the action plan previously mentioned have been successfully carried out and we are able to proceed to indentify and address other data abnormalies in the other features.

---
# 7. Outlier Detection

In this section, we will be identifying and addressing Outliers at the univariate level within our dataset. Outliers can cause issues such as skewing numerical summaries and visualisation. Therefore, it is important to identify and address the various outliers within the dataset appropriately.

In [ ]:
"""DAVI CA1 — Boxplots with Outlier Hover Showing 'flat_model'.

Renders one horizontal boxplot per numeric feature. Outlier points show the
associated 'flat_model' in the hover tooltip for quick cross-sectional insight.
"""

# =========================
# Setup Theme
# =========================
pio.templates.default = "plotly_white"


# =========================
# Guard for 'flat_model'
# =========================
if "flat_model" not in df.columns:
    raise KeyError("Column 'flat_model' not found in DataFrame.")


# =========================
# Select Numeric Columns
# =========================
num_cols = df.select_dtypes(include=["number"]).columns.tolist()
if not num_cols:
    raise ValueError("No numeric columns found in DataFrame.")


# =========================
# Render One Chart per Feature (with flat_model in hover)
# =========================
for col in num_cols:
    s = df[col].dropna()
    if s.empty:
        continue

    # Align flat_model to the same index as the numeric series
    flat_model_series = (
        df.loc[s.index, "flat_model"]
        .astype(str)
        .fillna("Unknown")
        .replace({"": "Unknown"})
    )

    d = pd.DataFrame({
        "feature": col,
        "value": s,
        "flat_model": flat_model_series,
    })

    fig = px.box(
        d,
        y="feature",                  # horizontal orientation
        x="value",
        points="outliers",            # show outlier markers only
        color_discrete_sequence=["#3B82F6"],
        custom_data=["flat_model"],   # pass flat_model to hovertemplate
    )

    fig.update_traces(
        boxmean=True,
        marker=dict(size=4, opacity=0.45, line=dict(width=0)),
        # Include flat_model in the hover; appears for outlier points
        hovertemplate="<b>%{y}</b><br>"
                      "Value: %{x:.4g}<br>"
                      "Flat model: %{customdata[0]}<extra></extra>",
    )

    fig.update_layout(
        title=dict(
            text=f"Distribution of <b>{col}</b>",
            x=0.5, xanchor="center",
            font=dict(size=20, family="Inter, Arial"),
        ),
        xaxis_title="Value",
        yaxis_title="",
        font=dict(size=12, family="Inter, Arial"),
        height=320,
        margin=dict(l=90, r=40, t=70, b=40),
        plot_bgcolor="#FAFAFA",
        paper_bgcolor="#FFFFFF",
        hoverlabel=dict(bgcolor="white", font_size=12),
        shapes=[dict(
            type="rect", xref="paper", yref="paper",
            x0=0, y0=0, x1=1, y1=1,
            line=dict(width=0),
            fillcolor="rgba(0,0,0,0)"
        )],
    )

    fig.update_xaxes(showgrid=True, gridcolor="rgba(0,0,0,0.08)", zeroline=False)
    fig.update_yaxes(showgrid=False, zeroline=False)

    fig.show()

With reference to the output of the code cell above, we are able to see that there are outliers for both the 'floor_area_sqm' and 'resale_price' data column. However, if we observe the box plot carefully, we are able to see that the outliers for both features beyond the upper quartile generally is in a consistent flow, this implies that these data are still somewhat consistent and most likely legitimate data. However, there are extreme outliers within these two features. Therefore, we will need to conduct further analysis to determine if these extreme outliers are illegitimate or legitimate values.

---
## 7.1 Floor Area Outlier Analysis

In this sub-section, we will be analysing the 'floor_area_sqm' feature to deterine whether the outliers are legitimate or illegitimate data. Understanding the importance of ensuring data integrity and quality, we will be inspecting the data from a threshold of '200' as based off the boxplot, that is where the outliers is no longer as consistent. We will be using the following formula to determine if the outliers are legitimate or illegitimate with refernece to the dataset's population.

---
**Peer Mean (Local Mean Floor Area)**
$$
\mu_{\text{peer}} = \frac{1}{n} \sum_{i=1}^{n} x_i
$$
Where:
- $x_i$ = Floor Area (sqm) of Each Flat Within the Same Town and Block
- $n$ = Number of Peer Flats in that Block
---
**Peer Standard Deviation**
$$
\sigma_{\text{peer}} = \sqrt{\frac{1}{n - 1} \sum_{i=1}^{n} (x_i - \mu_{\text{peer}})^2}
$$
Where:
- $μ_{\text{peer}}$ = Mean Floor Area of Peer Flats
- $x_i$ = Floor Area of the $i$-th Flat in the Same Block
---
**Percentage Difference from Peer Mean**
$$
\text{AreaDiff\%} = \frac{x_{\text{target}} - \mu_{\text{peer}}}{\mu_{\text{peer}}} \times 100
$$
Where:
- $x_{\text{target}}$ = Floor Area of the Current Flat being Evaluated

---
**Local Z-Score (Contextual Standard Score)**
$$
z_{\text{local}} =
\begin{cases}
\dfrac{x_{\text{target}} - \mu_{\text{peer}}}{\sigma_{\text{peer}}}, & \text{if } \sigma_{\text{peer}} \neq 0 \\
0, & \text{if } \sigma_{\text{peer}} = 0
\end{cases}
$$
Where:
- $z_{\text{local}}$ = Number of Standard Deviations the Target Flat's Floor Area Deviates from its Local Block Mean
---
With reference to the mathematical formulas which we will be utilising indicated above, we will proceed to conduct the outlier evaluation.

In [ ]:
"""DAVI CA1 — Contextual Outlier Analysis: Floor Area (Town + Block Scoped).

Flags unusually large floor areas by comparing each unit against peers
in the SAME town AND SAME block. Only groups with >= 3 peers are used.

Outputs a compact table with local mean/std, % difference, and z-score,
sorted by largest positive % difference first.
"""

# =========================
# Parameters & Guards
# =========================
COL_AREA = "floor_area_sqm"
TOWN_COL = "town"
BLOCK_COL = "block"
MIN_PEERS = 3
THRESHOLD = 200.0

for c in [COL_AREA, TOWN_COL, BLOCK_COL]:
    if c not in df.columns:
        raise KeyError(f"Required column '{c}' not found in DataFrame.")

# numeric safety
area_series = pd.to_numeric(df[COL_AREA], errors="coerce")


# =========================
# Peer Stats per (town, block)
# =========================
peer_stats = (
    df.assign(__area=area_series)
      .groupby([TOWN_COL, BLOCK_COL], dropna=False)["__area"]
      .agg(local_mean_area="mean", local_std_area="std", local_count="size")
      .reset_index()
)

# keep only groups with sufficient peers
peer_stats = peer_stats[peer_stats["local_count"] >= MIN_PEERS]


# =========================
# Focus on Large-area Candidates
# =========================
cands = df.loc[area_series > THRESHOLD, [
    TOWN_COL, BLOCK_COL, "flat_type", "flat_model", COL_AREA, "resale_price"
]].copy()

if cands.empty:
    print(f"No flats above threshold (> {THRESHOLD} sqm).")
else:
    # join peer stats
    ctx = (
        cands.merge(peer_stats, on=[TOWN_COL, BLOCK_COL], how="inner")
            .rename(columns={COL_AREA: "floor_area_sqm"})
    )

    # safe z-score (avoid divide-by-zero)
    std_safe = ctx["local_std_area"].replace(0, np.nan)
    ctx["local_z_score"] = (ctx["floor_area_sqm"] - ctx["local_mean_area"]) / std_safe
    ctx["local_z_score"] = ctx["local_z_score"].fillna(0.0)

    # percentage difference vs peers
    ctx["area_diff_%"] = (ctx["floor_area_sqm"] - ctx["local_mean_area"]) / ctx["local_mean_area"] * 100.0

    # rounding for presentation
    ctx["local_mean_area"] = ctx["local_mean_area"].round(2)
    ctx["local_std_area"] = ctx["local_std_area"].round(2)
    ctx["area_diff_%"] = ctx["area_diff_%"].round(2)
    ctx["local_z_score"] = ctx["local_z_score"].round(2)

    # reorder columns
    cols = [
        TOWN_COL, BLOCK_COL, "flat_type", "flat_model",
        "floor_area_sqm", "resale_price",
        "local_count", "local_mean_area", "local_std_area",
        "area_diff_%", "local_z_score",
    ]
    context_df_area = ctx.loc[:, cols].sort_values("area_diff_%", ascending=False, ignore_index=True)

    print(f"Large flats found (> {THRESHOLD} sqm): {len(context_df_area)}")
    display(style_count_red_flag(context_df_area, count_col="area_diff_%"))

With reference to the output of the DataFrame above, we will proceed with the following action.

**Action Plan**
- Proceed with Winsorization (capping) the Floor Area at 200 sqm for data with more than 150% area difference.

**Justification**
- Floor Area which are more than 2.5 times of other units within the same town and block is extremely unrealistic in Singapore's context due to the fact that HDBs in Singapore are at most 2 storey tall.

With the action plan and justification indicated above, we will proceed to conduct the winsorization operation with the data.

---
### 7.1.1 Addressing Outliers in Floor Area

As indicated in the previous sub-section, we will proceed to conduct the winsorization operation on the data so as to keep the Data but still not cause a major skew in our HDB dataset. The winsorization operation will be conducted in the code cell below.

In [ ]:
"""DAVI CA1 — Conditional Winsorisation for Floor Area (No Precomputed 'area_diff_%' Required).

Caps 'floor_area_sqm' to 200 sqm ONLY for records whose contextual deviation
vs peers in the SAME (town, block) exceeds +150%. Peer groups require at least
MIN_PEERS observations to be considered stable.

This cell recomputes the peer statistics and deviation on the fly, identifies the
rows to cap, applies the cap in-place, and reports before/after summaries.
"""

# =========================
# Parameters & Guards
# =========================
COL_AREA   = "floor_area_sqm"
COL_TOWN   = "town"
COL_BLOCK  = "block"
THRESHOLD_AREA = 200.0
THRESHOLD_DIFF = 150.0
MIN_PEERS      = 3

for c in (COL_AREA, COL_TOWN, COL_BLOCK):
    if c not in df.columns:
        raise KeyError(f"Required column '{c}' not found in DataFrame.")

# Ensure numeric area
df[COL_AREA] = pd.to_numeric(df[COL_AREA], errors="coerce")

# =========================
# Compute Peer Stats per (town, block)
# =========================
area_series = df[COL_AREA]
peer_stats = (
    df.assign(__area=area_series)
      .groupby([COL_TOWN, COL_BLOCK], dropna=False)["__area"]
      .agg(local_mean_area="mean", local_std_area="std", local_count="size")
      .reset_index()
)

# Use only sufficiently large groups
peer_stats = peer_stats.loc[peer_stats["local_count"] >= MIN_PEERS].copy()

# =========================
# Recompute Contextual Deviation and Find Rows to Cap
# =========================
# Work on candidates above the hard area threshold
cands = df.loc[area_series > THRESHOLD_AREA, [COL_TOWN, COL_BLOCK, COL_AREA]].copy()
if cands.empty or peer_stats.empty:
    print("No eligible rows for conditional winsorisation (no candidates or no stable peer groups).")
else:
    cands = (
        cands.merge(peer_stats, on=[COL_TOWN, COL_BLOCK], how="inner")
    )

    if cands.empty:
        print("No candidates belong to a stable (town, block) peer group; nothing to cap.")
    else:
        # % difference vs peer mean (avoid /0)
        with np.errstate(divide="ignore", invalid="ignore"):
            area_diff_pct = (cands[COL_AREA] - cands["local_mean_area"]) / cands["local_mean_area"] * 100.0
        area_diff_pct = area_diff_pct.replace([np.inf, -np.inf], np.nan).fillna(0.0)

        # Identify indices in the ORIGINAL df to cap
        # Merge preserved the original index alignment because we sliced from df first;
        # therefore cands.index corresponds to df.index for those rows.
        idx_to_cap = cands.index[(area_diff_pct > THRESHOLD_DIFF)].tolist()

        # =========================
        # Summary (Before)
        # =========================
        total_rows = len(df)
        before_stats = {
            "Max Floor Area (Before)": float(df[COL_AREA].max()) if len(df) else np.nan,
            "Rows Above 200 sqm (Before)": int((df[COL_AREA] > THRESHOLD_AREA).sum()),
            "Rows Eligible to Cap": int(len(idx_to_cap)),
            "Total Rows": int(total_rows),
        }

        # =========================
        # Apply Winsorisation
        # =========================
        if idx_to_cap:
            df.loc[idx_to_cap, COL_AREA] = THRESHOLD_AREA

        # =========================
        # Summary (After)
        # =========================
        after_stats = {
            "Max Floor Area (After)": float(df[COL_AREA].max()) if len(df) else np.nan,
            "Rows Above 200 sqm (After)": int((df[COL_AREA] > THRESHOLD_AREA).sum()),
            "Rows Winsorised": int(len(idx_to_cap)),
        }

        summary = pd.DataFrame(
            [{"Metric": k, "Value": (f"{v:,.2f}" if isinstance(v, (int, float)) else v)}
             for k, v in {**before_stats, **after_stats}.items()]
        )

        try:
            display(style_full_green(summary))
        except NameError:
            display(summary)

With reference to the code cell above, we are able to determine that the value have been successfully adjusted to 200 sqm. We will now proceed to conduct the next outlier analysis and addressment if needed.

---
## 7.2 Resale Price Outlier Analysis

In this sub-section, we will be analysing the 'resale_price' feature to determine whether the outliers are legitimate or illegitimate data. Ensuring the accuracy and integrity of the dataset is crucial for meaningful analysis, particularly since resale price is a key variable influencing housing affordability and market valuation. Based on the boxplot, prices above **\$1.5 million** exhibit irregular patterns and are considered potential outliers. Therefore, we will begin our contextual inspection at this threshold. The following mathematical formulas will be used to evaluate whether these outliers are legitimate or anomalous, by comparing each flat's resale price against other units within the same town and block.

---

**Peer Mean (Local Mean Resale Price)**
$$
\mu_{\text{peer}} = \frac{1}{n} \sum_{i=1}^{n} x_i
$$
Where:
- $x_i$ = Resale Price of Each Flat Within the Same Town and Block  
- $n$ = Number of Peer Flats in that Block  

---

**Peer Standard Deviation**
$$
\sigma_{\text{peer}} = \sqrt{\frac{1}{n - 1} \sum_{i=1}^{n} (x_i - \mu_{\text{peer}})^2}
$$
Where:
- $μ_{\text{peer}}$ = Mean Resale Price of Peer Flats  
- $x_i$ = Resale Price of the $i$-th Flat in the Same Block  

---

**Percentage Difference from Peer Mean**
$$
\text{PriceDiff\%} = \frac{x_{\text{target}} - \mu_{\text{peer}}}{\mu_{\text{peer}}} \times 100
$$
Where:
- $x_{\text{target}}$ = Resale Price of the Current Flat being Evaluated  

---

**Local Z-Score (Contextual Standard Score)**
$$
z_{\text{local}} =
\begin{cases}
\dfrac{x_{\text{target}} - \mu_{\text{peer}}}{\sigma_{\text{peer}}}, & \text{if } \sigma_{\text{peer}} \neq 0 \\
0, & \text{if } \sigma_{\text{peer}} = 0
\end{cases}
$$
Where:
- $z_{\text{local}}$ = Number of Standard Deviations the Target Flat’s Resale Price Deviates from its Local Block Mean  

---

With reference to the formulas above, we will now proceed to evaluate high-priced flats and assess whether these observations represent legitimate market anomalies or data inconsistencies that warrant further investigation.


In [ ]:
"""DAVI CA1 — Contextual Outlier Analysis: High Resale Prices (Town + Block Scoped).

Compares each high-priced flat (> THRESHOLD SGD) against peers in the SAME town
AND SAME block. Only (town, block) groups with at least MIN_PEERS observations
are considered to ensure stable peer statistics.

Emits:
    • local_count, local_mean_price, local_std_price
    • price_diff_%   (relative to local mean)
    • local_z_score  (standard score vs. local mean)

Sorted by largest positive % difference first.
"""

# =========================
# Parameters & Guards
# =========================
COL_PRICE  = "resale_price"
COL_TOWN   = "town"
COL_BLOCK  = "block"
MIN_PEERS  = 3
THRESHOLD  = 1_500_000
TOP_N      = None

for col_needed in (COL_PRICE, COL_TOWN, COL_BLOCK):
    if col_needed not in df.columns:
        raise KeyError(f"Required column '{col_needed}' not found in DataFrame.")

# Ensure numeric price
price_series = pd.to_numeric(df[COL_PRICE], errors="coerce")


# =========================
# Peer Stats Per (town, block)
# =========================
peer_stats = (
    df.assign(__price=price_series)
      .groupby([COL_TOWN, COL_BLOCK], dropna=False)["__price"]
      .agg(local_mean_price="mean",
           local_std_price="std",
           local_count="size")
      .reset_index()
)

# Keep only sufficiently large peer groups
peer_stats = peer_stats.loc[peer_stats["local_count"] >= MIN_PEERS].copy()


# =========================
# Candidate Flats Where Price > THRESHOLD
# =========================
cands = df.loc[price_series > THRESHOLD, [
    COL_TOWN, COL_BLOCK, "flat_type", "flat_model", "floor_area_sqm", COL_PRICE
]].copy()

if cands.empty:
    print(f"No flats above threshold (> ${THRESHOLD:,.0f}).")
else:
    # Join peer stats
    ctx = (
        cands.merge(peer_stats, on=[COL_TOWN, COL_BLOCK], how="inner")
            .rename(columns={COL_PRICE: "resale_price"})
    )

    if ctx.empty:
        print("No high-priced candidates have sufficient peers within the same (town, block).")
    else:
        # =========================
        # Metrics (Safe Division)
        # =========================
        with np.errstate(divide="ignore", invalid="ignore"):
            ctx["local_z_score"] = (
                (ctx["resale_price"] - ctx["local_mean_price"]) / ctx["local_std_price"]
            )
            ctx["price_diff_%"] = (
                (ctx["resale_price"] - ctx["local_mean_price"]) / ctx["local_mean_price"] * 100.0
            )

        # Handle zero/NaN std or mean gracefully
        ctx["local_z_score"] = ctx["local_z_score"].replace([np.inf, -np.inf], np.nan).fillna(0.0)
        ctx["price_diff_%"]  = ctx["price_diff_%"].replace([np.inf, -np.inf], np.nan).fillna(0.0)

        # Round for presentation
        ctx["local_mean_price"] = ctx["local_mean_price"].round(2)
        ctx["local_std_price"]  = ctx["local_std_price"].round(2)
        ctx["price_diff_%"]     = ctx["price_diff_%"].round(2)
        ctx["local_z_score"]    = ctx["local_z_score"].round(2)


        # =========================
        # Reorder & Display
        # =========================
        cols = [
            COL_TOWN, COL_BLOCK, "flat_type", "flat_model", "floor_area_sqm",
            "resale_price",
            "local_count", "local_mean_price", "local_std_price",
            "price_diff_%", "local_z_score",
        ]
        context_df_price = (
            ctx.loc[:, cols]
               .sort_values("price_diff_%", ascending=False, ignore_index=True)
        )

        if TOP_N:
            context_df_price = context_df_price.head(TOP_N)

        print(f"High-priced flats found (> ${THRESHOLD:,.0f}): {len(context_df_price)}")
        display(style_count_red_flag(context_df_price, count_col="price_diff_%"))

With reference to the output of the DataFrame above, we will proceed with the following action.

**Action Plan**
- Proceed with Winsorization (capping) the Resale Price at 1.5 Million dollars for data with more than 150% price difference.

**Justification**
- Resale Price which are more than 2.5 times of other units within the same town and block is extremely unrealistic in Singapore's context due to the similarity in unit specifications and location.

With the action plan and justification indicated above, we will proceed to conduct the winsorization operation with the data.

---
### 7.1.2 Addressing Outliers in Resale Price

As indicated in the previous sub-section, we will proceed to conduct the winsorization operation on the data so as to keep the Data but still not cause a major skew in our HDB dataset. The winsorization operation will be conducted in the code cell below.

In [ ]:
"""DAVI CA1 — Conditional Winsorisation for Price (No Precomputed 'price_diff_%' Required).

Caps 'resale_price' to $1.5M ONLY for records whose contextual deviation
vs peers in the SAME (town, block) exceeds +150%. Peer groups require at
least MIN_PEERS observations to be considered stable.

This cell recomputes peer statistics on the fly, identifies rows to cap,
applies the cap in-place, and reports before/after summaries.
"""

# =========================
# Parameters & Guards
# =========================
COL_PRICE       = "resale_price"
COL_TOWN        = "town"
COL_BLOCK       = "block"
THRESHOLD_PRICE = 1_500_000.0
THRESHOLD_DIFF  = 150.0
MIN_PEERS       = 3

for c in (COL_PRICE, COL_TOWN, COL_BLOCK):
    if c not in df.columns:
        raise KeyError(f"Required column '{c}' not found in DataFrame.")

# Ensure numeric price
df[COL_PRICE] = pd.to_numeric(df[COL_PRICE], errors="coerce")

# =========================
# Compute peer stats per (town, block)
# =========================
price_series = df[COL_PRICE]
peer_stats = (
    df.assign(__price=price_series)
      .groupby([COL_TOWN, COL_BLOCK], dropna=False)["__price"]
      .agg(local_mean_price="mean", local_std_price="std", local_count="size")
      .reset_index()
)

# Use only sufficiently large groups
peer_stats = peer_stats.loc[peer_stats["local_count"] >= MIN_PEERS].copy()

# =========================
# Recompute contextual deviation and find rows to cap
# =========================
# Work on candidates above the hard price threshold
cands = df.loc[price_series > THRESHOLD_PRICE, [COL_TOWN, COL_BLOCK, COL_PRICE]].copy()
if cands.empty or peer_stats.empty:
    print("No eligible rows for conditional winsorisation (no candidates or no stable peer groups).")
else:
    # Preserve original index for safe write-back after merge
    cands["__idx"] = cands.index

    cands = cands.merge(peer_stats, on=[COL_TOWN, COL_BLOCK], how="inner")

    if cands.empty:
        print("No high-priced candidates belong to a stable (town, block) peer group; nothing to cap.")
    else:
        # % difference vs peer mean (avoid divide-by-zero)
        with np.errstate(divide="ignore", invalid="ignore"):
            price_diff_pct = (cands[COL_PRICE] - cands["local_mean_price"]) / cands["local_mean_price"] * 100.0
        price_diff_pct = price_diff_pct.replace([np.inf, -np.inf], np.nan).fillna(0.0)

        # Identify original df indices to cap
        idx_to_cap = cands.loc[price_diff_pct > THRESHOLD_DIFF, "__idx"].tolist()

        # =========================
        # Summary (Before)
        # =========================
        total_rows = len(df)
        before_stats = {
            "Max Price (Before)": float(df[COL_PRICE].max()) if len(df) else np.nan,
            "Rows Above $1.5M (Before)": int((df[COL_PRICE] > THRESHOLD_PRICE).sum()),
            "Rows Eligible to Cap": int(len(idx_to_cap)),
            "Total Rows": int(total_rows),
        }

        # =========================
        # Apply Winsorisation (in-place)
        # =========================
        if idx_to_cap:
            df.loc[idx_to_cap, COL_PRICE] = THRESHOLD_PRICE

        # =========================
        # Summary (After)
        # =========================
        after_stats = {
            "Max Price (After)": float(df[COL_PRICE].max()) if len(df) else np.nan,
            "Rows Above $1.5M (After)": int((df[COL_PRICE] > THRESHOLD_PRICE).sum()),
            "Rows Winsorised": int(len(idx_to_cap)),
        }

        summary = pd.DataFrame(
            [{"Metric": k, "Value": (f"{v:,.2f}" if isinstance(v, (int, float)) else v)}
             for k, v in {**before_stats, **after_stats}.items()]
        )

        try:
            display(style_full_green(summary))
        except NameError:
            display(summary)

With reference to the code cell above, we are able to determine that the value have been successfully adjusted to 1.5 Million. We will now proceed to export the cleaned data.

---
# 8. Data Export

In this section, we will be exporting the data into CSV so that we are able to utilise this cleaned and processed data for Tableau visualisation purposes. This cleaned dataset will also be subsequently submitted for review as part of the assessment.

In [ ]:
"""DAVI CA1 — Export Cleaned Dataset.

Exports the fully cleaned and standardised HDB Resale Flat Prices dataset
to the predefined CLEAN_CSV path for downstream analysis and visualisation.
"""

# =========================
# Export DataFrame
# =========================
df.to_csv(CLEAN_CSV, index=False, encoding="utf-8-sig")


# =========================
# Verify Export
# =========================
file_size_mb = os.path.getsize(CLEAN_CSV) / (1024 * 1024)
summary = pd.DataFrame([
    {"Metric": "Export Path", "Value": str(CLEAN_CSV)},
    {"Metric": "Rows Exported", "Value": f"{len(df):,}"},
    {"Metric": "Columns", "Value": f"{len(df.columns):,}"},
    {"Metric": "File Size (MB)", "Value": f"{file_size_mb:.2f}"},
])

try:
    display(style_full_green(summary))
except NameError:
    display(summary)

With reference to the code cell above, we are able to determine that the cleaned dataset have been successfully exported into a new CSV file and we are ready to utilise this CSV file for Data Visualisation in Tableau.